In [ ]:
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

import pandas as pd
import time

In [ ]:
chrome_options = Options()

chrome_options.add_experimental_option("detach", True)
chrome_options.add_experimental_option("excludeSwitches",["enable-logging"])

driver = webdriver.Chrome(options=chrome_options)

#base_url = 'https://www.metacritic.com/browse/game/pc/all/all-time/userscore/?releaseYearMin=1958&releaseYearMax=2025&platform=pc&page=1'
#url = base_url
#driver.get(url)
#driver.maximize_window()

soup = BeautifulSoup(driver.page_source, 'html.parser')
title_nodes = soup.select('div.c-finderProductCard_title[data-title]')
titles = [n['data-title'].strip() for n in title_nodes]
#data_rows = soup.find_all('div', class_='c-finderProductCard_title') #find_all은 리스트로 가져옴
# data_rows = parent.find_all('h3', class_='c-finderProductCard_titleHeading')
BASE = ("https://www.metacritic.com/browse/game/pc/all/all-time/userscore/"
        "?releaseYearMin=1958&releaseYearMax=2025&platform=pc&page={page}")

all_names = []

for page in range(1,76):  # 1 ~ 75
    url = BASE.format(page=page)
    driver.get(url)
    driver.maximize_window()

    # 목록 로드 대기 (타이틀 div가 뜰 때까지)
    WebDriverWait(driver, 12).until(
        EC.presence_of_all_elements_located(
            (By.CSS_SELECTOR, 'div.c-finderProductCard_title[data-title]')
        )
    )

    # data-title 속성만 추출
    elems = driver.find_elements(By.CSS_SELECTOR, 'div.c-finderProductCard_title[data-title]')
    names = [e.get_attribute('data-title').strip() for e in elems]
    all_names.extend(names)

    print(f"page {page}: {len(names)} items (total {len(all_names)})")
    time.sleep(0.5)  # 서버 로딩 여유

# 저장(선택)
pd.DataFrame({'title': all_names}).to_csv('../outputs/metacritic_pc_userscore_green.csv', index=False)
print("done:", len(all_names))